In [ ]:
import duckdb
import numpy as np
import pandas as pd
from math import log10

In [ ]:
def apply_special_ffill(df: pd.DataFrame, ticker_col: str = "ticker", date_col: str = "date") -> pd.DataFrame:
    
    con = duckdb.connect()
    con.register("raw", df)

    result = con.execute(f"""
        WITH lagged AS (
            SELECT
                *,
                LAG(C)  OVER w AS prev_close,
                LAG(O)  OVER w AS prev_open,   -- 前日の有効性確認用
                LAG(Vo) OVER w AS prev_volume,  -- 前日の有効性確認用

                -- 補完対象フラグ: 自分が全NULL & 前日closeが非NULL
                CASE
                    WHEN O  IS NULL
                     AND H  IS NULL
                     AND L  IS NULL
                     AND C  IS NULL
                     AND Vo IS NULL
                     AND LAG(C) OVER w IS NOT NULL
                    THEN TRUE
                    ELSE FALSE
                END AS is_filled
            FROM raw
            WINDOW w AS (
                PARTITION BY {ticker_col}
                ORDER BY {date_col}
                ROWS BETWEEN 1 PRECEDING AND CURRENT ROW
            )
        )
        SELECT
            {ticker_col},
            {date_col},
            CASE WHEN is_filled THEN prev_close ELSE O  END AS O,
            CASE WHEN is_filled THEN prev_close ELSE H  END AS H,
            CASE WHEN is_filled THEN prev_close ELSE L  END AS L,
            CASE WHEN is_filled THEN prev_close ELSE C  END AS C,
            CASE WHEN is_filled THEN 0.0        ELSE Vo END AS Vo,
            is_filled
        FROM lagged
        ORDER BY {ticker_col}, {date_col}
    """).df()

    con.close()
    return result


# ──────────────────────────────────────────────────────────
# 動作確認
# ──────────────────────────────────────────────────────────
if __name__ == "__main__":
    nan = float("nan")
    sample = pd.DataFrame({
        "ticker": ["A"] * 6 + ["B"] * 4,
        "date": list(pd.date_range("2024-01-01", periods=6, freq="B"))
              + list(pd.date_range("2024-01-01", periods=4, freq="B")),
        #          通常     通常    NULL    NULL連続  通常    通常
        "open":  [100.0, 102.0,  nan,    nan,    105.0, 108.0,
                  200.0, 202.0,  nan,    205.0],
        "high":  [103.0, 104.0,  nan,    nan,    107.0, 110.0,
                  203.0, 205.0,  nan,    208.0],
        "low":   [ 99.0, 101.0,  nan,    nan,    104.0, 106.0,
                  198.0, 200.0,  nan,    203.0],
        "close": [102.0, 101.0,  nan,    nan,    106.0, 109.0,
                  201.0, 202.0,  nan,    206.0],
        "volume":[1000,   800,   nan,    nan,    1200,  900,
                   500,   600,   nan,    700],
    })

    print("=== Before ===")
    print(sample.to_string(index=False))

    out = apply_special_ffill(sample)

    print("\n=== After ===")
    print(out.to_string(index=False))